[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/geometry/sundial/sundial.ipynb)

# Sundials

A sundial tells the time by a shadow. Its gnomon has an edge, the style, that points at the celestial pole, the spot in the sky the earth's axis points to; the sky, and the sun with it, turns once a day about that axis, and the style's shadow turns with it, along a fixed line of the dial for each hour. The tip of the gnomon, the nodus, casts a point whose path over a day tells the season. Both come out of two turns of one direction, and the season's path is a conic: the dial cut through the cone the day's sunbeams sweep about the pole.

In [ ]:
# The repository root on the path, for numga and the examples; in Colab, fetch the repository first.
import sys
from pathlib import Path

if "google.colab" in sys.modules:
    root = Path("/content/numga")
    if not root.exists():
        import subprocess
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/EelcoHoogendoorn/numga.git", str(root)], check=True)
else:
    root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "numga").is_dir() and (p / "examples").is_dir())
sys.path.insert(0, str(root))

In [ ]:
%matplotlib inline
import numpy as np

from numga import NumpyContext
from numga.algebras import VGA3D as ga
from examples.animation import save_figure, show_animation
from examples.geometry.sundial import render

mv = NumpyContext(ga).multivector
Scalar = ga.gatype.scalar()
Vector = ga.gatype.vector()
Bivector = ga.gatype.bivector()
Rotor = ga.gatype.rotor()
# A quadratic form on rays.
Cone = ga.gatype((Vector, Vector))                         # Vector <- Vector
east, north, up = mv.basis()                               # [] Vector each
west = -east

# The dial's latitude, the height of the nodus above the dial, and the tilt of the earth's axis, the sun's
# declination at the solstices.
latitude = np.radians(52.0)
height = 1.0
tilt = np.radians(23.44)
# The seasons drawn, a whole number of months from an equinox up to a solstice on either side; the hours
# drawn; the declinations each hour line runs through, from solstice to solstice; and the samples of a day.
months = 3
hours = np.arange(6, 19)
declination_samples = 25
day_samples = np.arange(3, 21.01, 0.05)
# The frames of one year, the hours a frame's rays are drawn at, and the duration of a frame.
frames_count = 48
ray_hours = np.arange(4, 21, 0.25)
duration_ms = 90


def hour_angle(hours: np.ndarray) -> np.ndarray:
    """The sun's hour angle at each hour of local solar time: a fifteenth of a quarter turn per hour from
    noon, westwards in the afternoon."""
    return np.radians(15.0 * (hours - 12))

## 1. The sun's two turns

The celestial pole is north raised towards the zenith by the latitude, and the celestial equator's highest point lies a quarter turn further. The sun at noon on the equinoxes stands there. Any other sun is that point turned twice: towards the pole by the declination, the season's angle north or south of the equator, then about the pole towards the west by the hour angle, a twenty-fourth of a turn per hour. At noon the sun stands a quarter turn less the latitude, plus the declination, above the horizon.

In [ ]:
def turn(plane: Bivector, angle: np.ndarray) -> Rotor:
    """The turn by an angle in a plane through the origin."""
    return (plane * (-angle / 2)).exp()


def sky(latitude: float) -> tuple[Vector, Vector]:
    """The celestial pole, north raised towards up by the latitude, and the celestial equator's highest
    point, a quarter turn further."""
    return turn(north ^ up, latitude) >> north, turn(north ^ up, latitude + np.pi / 2) >> north


def sun(pole: Vector, equator: Vector, declinations: np.ndarray, hours: np.ndarray) -> Vector:
    """The direction to the sun at each declination and hour angle: the equator's highest point turned
    towards the pole by the declination, then about the pole towards the west by the hour angle."""
    noon = turn(equator ^ pole, declinations) >> equator                      # [...] Vector
    return turn(equator ^ west, hours) >> noon                                # [...] Vector


pole, equator = sky(latitude)
solstices_and_equinox = np.array([-tilt, 0.0, tilt])
noon_suns = sun(pole, equator, solstices_and_equinox, np.zeros(3))            # [seasons] Vector
print("noon heights, winter, equinox, summer:", np.degrees(np.arcsin(render.scalars(noon_suns | up))).round(2))

## 2. The hour lines

The nodus casts its shadow where the ray from the sun past the nodus meets the dial. The style runs through the nodus towards the pole, and the plane through the style and the sun at a given hour holds the sun at that hour on every day of the year: the declination turns the sun within that plane. So each hour's shadow, whatever the season, falls along one line of the dial through the foot of the style. Measured from the noon line, the hour lines crowd towards noon and spread towards morning and evening.

In the notation of gnomonics a horizontal dial's hour line makes the angle $\arctan(\sin\varphi \tan H)$ with the noon line, at latitude $\varphi$ and hour angle $H$.

In [ ]:
def shadow(nodus: Vector, suns: Vector) -> Vector:
    """Where the shadow of the nodus falls on the dial, the horizontal plane through the origin: from the
    nodus, away from the sun, as far as the dial."""
    return nodus - suns * ((nodus | up) / (suns | up))                         # [...] Vector


nodus = up * height                                                            # [] Vector
# Where the style meets the dial.
foot = shadow(nodus, pole)                                                     # [] Vector
declinations = np.linspace(-tilt, tilt, declination_samples)                    # [samples]
hour_suns = sun(pole, equator, declinations[None, :], hour_angle(hours)[:, None])   # [hours, samples] Vector
hour_lines = shadow(nodus, hour_suns)                                          # [hours, samples] Vector
spread = render.bearings(hour_lines - foot)                                    # [hours, samples]
print("hour line bearings from the noon line, in degrees:", np.degrees(spread[:, -1]).round(1))

## 3. The seasons are conics

On a given day the ray from the nodus to the sun keeps its angle to the pole, a quarter turn less the declination, so the day's rays sweep a cone about the pole. A cone is a quadratic form on rays, zero on the rays along it: `pole * (pole | Vector) - sin(declination)**2 * Vector`. The dial cuts it, and the shadow path is the cut, a conic. The form only sees the declination squared, so its other nappe holds the rays of the opposite declination: a summer day and the winter day as far from the equinox are the two branches of one hyperbola, drawn here in one colour. At the equinoxes the cone flattens into the plane of the celestial equator, and the shadow runs along a straight line.

In [ ]:
def cone(pole: Vector, declinations: np.ndarray) -> Cone:
    """The cone of rays at a declination's angle from the celestial equator: zero on a ray exactly when its
    angle to the pole is a quarter turn less the declination, or a quarter turn more."""
    return pole * (pole | Vector) - mv.scalar(np.sin(declinations)[..., None] ** 2) * Vector   # [...] Vector <- Vector


seasons = tilt * np.sin(np.pi / 2 * np.arange(-months, months + 1) / months)   # [seasons]
days = sun(pole, equator, seasons[:, None], hour_angle(day_samples)[None, :])  # [seasons, samples] Vector
rays = (nodus - shadow(nodus, days)).normalized()                              # [seasons, samples] Vector
print("largest value of a season's cone on its day's shadow rays:", np.abs(render.scalars(rays | cone(pole, seasons)[:, None](rays))).max())
save_figure(render.draw_dial(nodus, foot, seasons, days, hour_suns, hours), "sundial");

## 4. A year

Through the year the declination swings between the solstices, and the cone of the day's rays opens and closes about the pole: steep in summer, its cut with the dial a tight hyperbola near the gnomon; flat at the equinoxes, a straight line; shallow in winter, the hyperbola's far branch. On the left, the day's rays from beyond the nodus down to the dial; on the right, that day's shadow path among the seasons'.

In [ ]:
year_declinations = tilt * np.sin(np.linspace(0, 2 * np.pi, frames_count, endpoint=False))   # [frames]
year_rays = sun(pole, equator, year_declinations[:, None], hour_angle(ray_hours)[None, :])   # [frames, hours] Vector
year_days = sun(pole, equator, year_declinations[:, None], hour_angle(day_samples)[None, :])   # [frames, samples] Vector

show_animation(render.animate_year(year_declinations, year_rays, year_days, nodus, foot, seasons, days, hour_suns, hours),
               "sundial_year", duration_ms)

In [ ]:
# checks
# The noon sun stands a quarter turn less the latitude, plus the declination, above the horizon.
assert np.abs(np.arcsin(render.scalars(noon_suns | up)) - (np.pi / 2 - latitude + solstices_and_equinox)).max() < 1e-10
# Every shadow ray of a season's day lies on its cone, and each hour line on the angle of gnomonics.
assert np.abs(render.scalars(rays | cone(pole, seasons)[:, None](rays))).max() < 1e-10
expected = np.arctan2(np.sin(latitude) * np.sin(hour_angle(hours)), np.cos(hour_angle(hours)))
assert np.abs(np.sin(spread - expected[:, None])).max() < 1e-10